In [ ]:
# cjpe_qa_ollama.py
# ─────────────────────────────────────────────────────────────
# Generates 10 judgment-predictive QA pairs per case (Track A):
#   • 6 from FAC   (who/what/when/where/which/why)
#   • 2 from ARG_P (petitioner argument strength)
#   • 2 from ARG_R (respondent argument strength)
#
# Provider  : Ollama (local — unlimited, no API key needed)
# Model     : llama3.1 (8B)
# Input     : cjpe_track_A_clean.jsonl
# Output    : Track_A_qa_judgment_flat.jsonl
# Stops at  : 8,000 successfully processed docs
#
# Fixes:
#   - Escaped single quotes  \'  inside JSON strings
#   - Truncated responses (increased num_predict)
#   - 8-layer JSON rescue pipeline
#   - Regex key-value fallback as last resort
# ─────────────────────────────────────────────────────────────

import json
import os
import re
import time
import requests
from typing import Optional

# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
MODEL_ID        = "llama3.1"
OLLAMA_URL      = "http://localhost:11434/api/chat"

INPUT_PATH      = "cjpe_track_A_clean.jsonl"
OUTPUT_PATH     = "Track_A_qa_judgment_flat_OLLAMA.jsonl"
CHECKPOINT_PATH = "TrackAqa_checkpoint_OLLAMA.json"

TARGET_DOCS     = 8000

# ══════════════════════════════════════════════════════════════
# PROMPTS
# ══════════════════════════════════════════════════════════════

FAC_PROMPT = """You are a legal expert analyzing Indian court judgments.

JUDGMENT OUTCOME: {label_word}

FACTS:
{fac}

Generate exactly 6 question-answer pairs from these facts.
Use these question types in order: WHO, WHAT, WHEN, WHERE, WHICH, WHY.
Each answer MUST end with exactly one of: [FAVORS_PETITIONER] or [FAVORS_RESPONDENT] or [NEUTRAL]

IMPORTANT: Do NOT use apostrophes or single quotes in your response. Use full words instead.
Example: use "does not" instead of "doesn't", use "company" instead of "company's".

Reply with ONLY this JSON, no other text:
{{"Q1":"who question","A1":"answer [FAVORS_PETITIONER]","Q2":"what question","A2":"answer [FAVORS_RESPONDENT]","Q3":"when question","A3":"answer [NEUTRAL]","Q4":"where question","A4":"answer [SIGNAL]","Q5":"which question","A5":"answer [SIGNAL]","Q6":"why question","A6":"answer [SIGNAL]"}}

Replace [SIGNAL] with the correct signal tag for each answer."""

ARG_P_PROMPT = """You are a legal expert analyzing Indian court judgments.

JUDGMENT OUTCOME: {label_word}

PETITIONER ARGUMENTS:
{arg_p}

Generate exactly 2 question-answer pairs evaluating petitioner argument strength.
Each answer MUST end with exactly one of: [FAVORS_PETITIONER] or [FAVORS_RESPONDENT] or [NEUTRAL]

IMPORTANT: Do NOT use apostrophes or single quotes. Use full words only.

Reply with ONLY this JSON, no other text:
{{"Q1":"question about statute or precedent backing","A1":"answer [SIGNAL]","Q2":"question about burden of proof","A2":"answer [SIGNAL]"}}

Replace [SIGNAL] with the correct signal tag."""

ARG_R_PROMPT = """You are a legal expert analyzing Indian court judgments.

JUDGMENT OUTCOME: {label_word}

RESPONDENT ARGUMENTS:
{arg_r}

Generate exactly 2 question-answer pairs evaluating respondent argument strength.
Each answer MUST end with exactly one of: [FAVORS_PETITIONER] or [FAVORS_RESPONDENT] or [NEUTRAL]

IMPORTANT: Do NOT use apostrophes or single quotes. Use full words only.

Reply with ONLY this JSON, no other text:
{{"Q1":"question about respondent statute or precedent","A1":"answer [SIGNAL]","Q2":"question about undermining petitioner claim","A2":"answer [SIGNAL]"}}

Replace [SIGNAL] with the correct signal tag."""

# ══════════════════════════════════════════════════════════════
# PROMPT BUILDERS
# ══════════════════════════════════════════════════════════════

def label_word(label: int) -> str:
    return "ACCEPTED (Appeal Allowed)" if label == 1 else "REJECTED (Appeal Dismissed)"

def build_fac_prompt(fac: str, label: int) -> str:
    return FAC_PROMPT.format(label_word=label_word(label), fac=fac[:3000])

def build_arg_p_prompt(arg_p: str, label: int) -> str:
    return ARG_P_PROMPT.format(label_word=label_word(label), arg_p=arg_p[:1200])

def build_arg_r_prompt(arg_r: str, label: int) -> str:
    return ARG_R_PROMPT.format(label_word=label_word(label), arg_r=arg_r[:1200])

# ══════════════════════════════════════════════════════════════
# ROBUST JSON EXTRACTOR — 8-layer rescue pipeline
# ══════════════════════════════════════════════════════════════

def clean_raw(raw: str) -> str:
    """Pre-clean raw string before JSON parsing attempts."""
    # Fix escaped single quotes  \'  → remove the backslash
    raw = raw.replace("\\'", " ")
    # Fix escaped double quotes inside values  \"  → use a placeholder then restore
    # (only remove fences, not legitimate escapes inside strings)
    raw = re.sub(r"```(?:json)?", "", raw)
    return raw.strip()


def extract_json(raw: str) -> dict:
    if not raw:
        return {}

    # ── Pre-clean ─────────────────────────────────────────────
    raw = clean_raw(raw)

    # ── Step 1: extract first { ... } block ───────────────────
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        # Try regex fallback immediately if no braces found
        return regex_fallback(raw)
    raw = match.group(0)

    # ── Step 2: direct parse ──────────────────────────────────
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # ── Step 3: fix trailing commas ───────────────────────────
    cleaned = re.sub(r",\s*([}\]])", r"\1", raw)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # ── Step 4: collapse newlines inside values ────────────────
    cleaned = re.sub(
        r':\s*"(.*?)"(?=\s*[,}])',
        lambda m: ': "' + re.sub(r'[\n\r]+', ' ', m.group(1)) + '"',
        cleaned,
        flags=re.DOTALL
    )
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # ── Step 5: fix trailing commas again after collapse ──────
    cleaned = re.sub(r",\s*([}\]])", r"\1", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # ── Step 6: replace single quotes with double quotes ──────
    cleaned = cleaned.replace("'", '"')
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # ── Step 7: remove all control characters ─────────────────
    cleaned = re.sub(r'[\x00-\x1f\x7f]', ' ', cleaned)
    cleaned = re.sub(r",\s*([}\]])", r"\1", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # ── Step 8: regex key-value fallback ──────────────────────
    return regex_fallback(raw)


def regex_fallback(raw: str) -> dict:
    """
    Last resort: extract Q1/A1...Q6/A6 directly using regex.
    Works even when the JSON braces/commas are completely broken.
    """
    result  = {}

    # Pattern 1: standard  "Q1": "value"
    pattern = r'"(Q\d+|A\d+)"\s*:\s*"(.*?)"(?=\s*[,}\n]|$)'
    matches = re.findall(pattern, raw, re.DOTALL)
    for key, val in matches:
        result[key] = re.sub(r'[\n\r]+', ' ', val).strip()

    if result:
        return result

    # Pattern 2: relaxed — key: value without quotes
    pattern2 = r'(Q\d+|A\d+)\s*:\s*"?(.*?)"?(?=\s*(?:Q\d+|A\d+)\s*:|$)'
    matches2  = re.findall(pattern2, raw, re.DOTALL)
    for key, val in matches2:
        result[key] = re.sub(r'[\n\r]+', ' ', val).strip().strip('"').strip("'")

    return result


# ══════════════════════════════════════════════════════════════
# OLLAMA API CALL — with retry + backoff
# ══════════════════════════════════════════════════════════════

def call_ollama(prompt: str, retries: int = 5) -> dict:
    payload = {
        "model"   : MODEL_ID,
        "messages": [{"role": "user", "content": prompt}],
        "stream"  : False,
        "options" : {
            "temperature" : 0.1,
            "num_predict" : 2048,   # increased — prevents truncation
            "repeat_penalty": 1.1, # reduces repetitive garbage output
        }
    }

    for attempt in range(retries):
        try:
            resp = requests.post(OLLAMA_URL, json=payload, timeout=240)

            if resp.status_code == 404:
                print(f"\n  ❌ Ollama model '{MODEL_ID}' not found.")
                print(f"     Run:  ollama pull {MODEL_ID}")
                raise SystemExit(1)

            if resp.status_code in (502, 503, 504):
                wait = 20 * (attempt + 1)
                print(f"    ⏳ Server error ({resp.status_code}) — waiting {wait}s...")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            data   = resp.json()
            raw    = data["message"]["content"].strip()
            result = extract_json(raw)

            if result:
                return result
            else:
                print(f"    ⚠️  JSON extract failed (attempt {attempt+1}) "
                      f"— raw: {raw[:120]!r}")
                time.sleep(2)

        except requests.exceptions.ConnectionError:
            print(f"\n  ❌ Cannot connect to Ollama. Run:  ollama serve")
            raise SystemExit(1)

        except requests.exceptions.Timeout:
            print(f"    ⚠️  Timeout (attempt {attempt+1}) — retrying...")
            time.sleep(10)

        except SystemExit:
            raise

        except Exception as e:
            print(f"    ⚠️  Error (attempt {attempt+1}): {e}")
            time.sleep(5)

    return {}

# ══════════════════════════════════════════════════════════════
# FLATTEN QA DICT → list of individual QA records
# ══════════════════════════════════════════════════════════════

def flatten_qa(doc_id: str, qa_dict: dict, label: int,
               start_idx: int, source: str) -> list:
    records = []
    n = len([k for k in qa_dict if k.startswith("Q")])

    for i in range(1, n + 1):
        q = qa_dict.get(f"Q{i}", "").strip()
        a = qa_dict.get(f"A{i}", "").strip()

        if not q or not a:
            continue

        signal = "NEUTRAL"
        for tag in ["FAVORS_PETITIONER", "FAVORS_RESPONDENT", "NEUTRAL"]:
            if f"[{tag}]" in a:
                signal = tag
                a = a.replace(f"[{tag}]", "").strip()
                break

        records.append({
            "id"      : f"{doc_id}_Q{start_idx + i}",
            "doc_id"  : doc_id,
            "source"  : source,
            "question": q,
            "answer"  : a,
            "signal"  : signal,
            "label"   : label
        })

    return records

# ══════════════════════════════════════════════════════════════
# CHECKPOINT HELPERS
# ══════════════════════════════════════════════════════════════

def load_checkpoint() -> Optional[str]:
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH) as f:
            data = json.load(f)
            return data.get("last_doc_id")
    return None

def save_checkpoint(doc_id: str, processed: int):
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump({"last_doc_id": str(doc_id), "processed": processed}, f)

# ══════════════════════════════════════════════════════════════
# OLLAMA SANITY CHECK
# ══════════════════════════════════════════════════════════════

def check_ollama():
    print("  🔍 Checking Ollama connection...")
    try:
        resp    = requests.get("http://localhost:11434/api/tags", timeout=5)
        models  = [m["name"] for m in resp.json().get("models", [])]
        matched = [m for m in models if MODEL_ID in m]
        if not matched:
            print(f"\n  ❌ Model '{MODEL_ID}' not found.")
            print(f"     Available: {models}")
            print(f"     Run: ollama pull {MODEL_ID}")
            raise SystemExit(1)
        print(f"  ✅ Ollama running | Model '{MODEL_ID}' found\n")
    except requests.exceptions.ConnectionError:
        print(f"\n  ❌ Ollama is not running. Start with:  ollama serve")
        raise SystemExit(1)

# ══════════════════════════════════════════════════════════════
# LOAD DATA
# ══════════════════════════════════════════════════════════════

records = []
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

# Resume from checkpoint
last_doc_id       = load_checkpoint()
already_processed = 0

if last_doc_id is not None:
    ids = [str(r["id"]) for r in records]
    if last_doc_id in ids:
        resume_idx        = ids.index(last_doc_id) + 1
        already_processed = json.load(open(CHECKPOINT_PATH)).get("processed", resume_idx)
        records           = records[resume_idx:]
        print(f"  ♻️  Resuming after '{last_doc_id}'")
        print(f"  ✅ Already done  : {already_processed:,} docs")
        remaining_target  = TARGET_DOCS - already_processed
        if remaining_target <= 0:
            print(f"  ✅ Target already reached!")
            raise SystemExit(0)
        print(f"  📌 Still needed  : {remaining_target:,} docs\n")
    else:
        print(f"  ⚠️  Checkpoint not found — fresh start")
        already_processed = 0
else:
    print("  🚀 Fresh start")
    already_processed = 0

# ══════════════════════════════════════════════════════════════
# STARTUP
# ══════════════════════════════════════════════════════════════

check_ollama()

print(f"  📁 Input          : {INPUT_PATH}")
print(f"  💾 Output         : {OUTPUT_PATH}")
print(f"  🤖 Model          : {MODEL_ID} (Ollama local)")
print(f"  🎯 Target docs    : {TARGET_DOCS:,}")
print(f"  📊 QA per doc     : 10  (6 FAC + 2 ARG_P + 2 ARG_R)")
print(f"  📦 Total QA target: ~{TARGET_DOCS * 10:,} pairs\n")

# ══════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════

out_file   = open(OUTPUT_PATH, "a", encoding="utf-8")
processed  = already_processed
errors     = 0
start_time = time.time()

for rec in records:

    # ── EARLY STOP ────────────────────────────────────────────
    if processed >= TARGET_DOCS:
        print(f"\n  🎯 Target of {TARGET_DOCS:,} docs reached — stopping!")
        break

    doc_id = str(rec["id"])
    label  = int(rec["label"])
    fac    = rec.get("FAC",   "").strip()
    arg_p  = rec.get("ARG_P", "").strip()
    arg_r  = rec.get("ARG_R", "").strip()

    if not fac:
        errors += 1
        continue

    all_flat = []

    # ── FAC: 6 questions ──────────────────────────────────────
    fac_dict = call_ollama(build_fac_prompt(fac, label))
    if fac_dict:
        all_flat.extend(flatten_qa(doc_id, fac_dict, label,
                                   start_idx=0, source="FAC"))
    else:
        print(f"  ⚠️  FAC failed for {doc_id}")

    # ── ARG_P: 2 questions ────────────────────────────────────
    if arg_p:
        arg_p_dict = call_ollama(build_arg_p_prompt(arg_p, label))
        if arg_p_dict:
            all_flat.extend(flatten_qa(doc_id, arg_p_dict, label,
                                       start_idx=6, source="ARG_P"))
        else:
            print(f"  ⚠️  ARG_P failed for {doc_id}")

    # ── ARG_R: 2 questions ────────────────────────────────────
    if arg_r:
        arg_r_dict = call_ollama(build_arg_r_prompt(arg_r, label))
        if arg_r_dict:
            all_flat.extend(flatten_qa(doc_id, arg_r_dict, label,
                                       start_idx=8, source="ARG_R"))
        else:
            print(f"  ⚠️  ARG_R failed for {doc_id}")

    # ── Write + Checkpoint ────────────────────────────────────
    if all_flat:
        for flat_rec in all_flat:
            out_file.write(json.dumps(flat_rec, ensure_ascii=False) + "\n")
        out_file.flush()

        processed += 1
        save_checkpoint(doc_id, processed)

        if processed % 100 == 0:
            elapsed = time.time() - start_time
            per_doc = elapsed / max(processed - already_processed, 1)
            eta_hrs = ((TARGET_DOCS - processed) * per_doc) / 3600
            print(f"  ✅ {processed:>5}/{TARGET_DOCS} docs | "
                  f"~{processed*10:,} QA pairs | "
                  f"⏱ {per_doc:.1f}s/doc | "
                  f"ETA: {eta_hrs:.1f} hrs | "
                  f"last: {doc_id}")
    else:
        errors += 1
        print(f"  ⚠️  No QA for {doc_id}")

out_file.close()

# ══════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════
elapsed_total = (time.time() - start_time) / 3600

print("\n" + "=" * 60)
print("  QA GENERATION COMPLETE")
print("=" * 60)
print(f"  Docs processed : {processed:,} / {TARGET_DOCS:,}")
print(f"  Total QA pairs : ~{processed * 10:,}")
print(f"  Errors         : {errors}")
print(f"  Total time     : {elapsed_total:.2f} hrs")
print(f"  Output file    : {OUTPUT_PATH}")

print("\n  Sample output (first 5 QA pairs):")
with open(OUTPUT_PATH) as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        r = json.loads(line)
        print(f"\n  ── Record {i+1} ──────────────────────")
        print(f"  ID     : {r['id']}")
        print(f"  Source : {r['source']}")
        print(f"  Signal : {r['signal']}")
        print(f"  Label  : {r['label']} ({'ACCEPTED' if r['label']==1 else 'REJECTED'})")
        print(f"  Q: {r['question']}")
        print(f"  A: {r['answer']}")

  ♻️  Resuming after '2016_622'
  ✅ Already done  : 1,005 docs
  📌 Still needed  : 6,995 docs

  🔍 Checking Ollama connection...
  ✅ Ollama running | Model 'llama3.1' found

  📁 Input          : cjpe_track_A_clean.jsonl
  💾 Output         : Track_A_qa_judgment_flat_OLLAMA.jsonl
  🤖 Model          : llama3.1 (Ollama local)
  🎯 Target docs    : 8,000
  📊 QA per doc     : 10  (6 FAC + 2 ARG_P + 2 ARG_R)
  📦 Total QA target: ~80,000 pairs

